In [1]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

In [2]:
"""
Full Multi-Agent Research System

This module integrates all components of the research system:
- User clarification and scoping
- Research brief generation  
- Multi-agent research coordination
- Final report generation

The system orchestrates the complete research workflow from initial user
input through final report delivery.
"""

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END

from deepresearch.utils import get_today_str
from deepresearch.prompt import final_report_generation_prompt
from deepresearch.state import AgentState, AgentInputState
from deepresearch.agents.scoping_agent import clarify_with_user, write_research_brief
from deepresearch.agents.supervisor_agent import supervisor_agent

_DEFAULT_FINAL_REPORT_MODEL = "groq:qwen/qwen3-32b"

async def final_report_generation(state: AgentState, config: RunnableConfig):
    """
    Final report generation node.
    
    Synthesizes all research findings into a comprehensive final report
    """
    
    model_name = config.get("configurable", {}).get("final_report_model", _DEFAULT_FINAL_REPORT_MODEL)
    writer_model = init_chat_model(model_name, temperature=0.7)

    notes = state.get("notes", [])
    findings = "\n".join(notes)

    final_report_prompt = final_report_generation_prompt.format(
        research_brief=state.get("research_brief", ""),
        findings=findings,
        date=get_today_str()
    )

    final_report = await writer_model.ainvoke([HumanMessage(content=final_report_prompt)])
    
    return {
        "final_report": final_report.content, 
        "messages": ["Here is the final report: " + final_report.content],
    }

deep_researcher_builder = StateGraph(AgentState, input_schema=AgentInputState)

# Add workflow nodes
deep_researcher_builder.add_node("clarify_with_user", clarify_with_user)
deep_researcher_builder.add_node("write_research_brief", write_research_brief)
deep_researcher_builder.add_node("supervisor_subgraph", supervisor_agent)
deep_researcher_builder.add_node("final_report_generation", final_report_generation)

# Add workflow edges
deep_researcher_builder.add_edge(START, "clarify_with_user")
deep_researcher_builder.add_edge("write_research_brief", "supervisor_subgraph")
deep_researcher_builder.add_edge("supervisor_subgraph", "final_report_generation")
deep_researcher_builder.add_edge("final_report_generation", END)

# Compile the full workflow
agent = deep_researcher_builder.compile()

Loaded: gsk_uhqiMH...


In [3]:
from langchain_core.messages import HumanMessage
from utils import format_messages

import uuid

thread = {
    "configurable": {
        "thread_id": str(uuid.uuid4()),
        "recursion_limit": 50
    }
}

result = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content="Best IPL team")
        ]
    },
    config=thread
)

format_messages(result["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Best IPL team                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ Could you please clarify which season of the Indian Premier League you are referring to?                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
result = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content="Best IPL team throughout the ipl ")
        ]
    },
    config=thread
)


In [5]:
format_messages(result["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Best IPL team throughout the ipl                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ Could you please specify which season or time period of the Indian Premier League (IPL) you are referring to?   │
│ For example, are you looking for the best team in the tournament's overall history, or perhaps in a specific    │
│ season like 2024 or 2025?                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
result = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content="Best IPL team throughout the ipl from 2008-2026  ")
        ]
    },
    config=thread
)


In [8]:
format_messages(result["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Best IPL team throughout the ipl from 2008-2026                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ Which years specifically (e.g. 2008-2010, 2015-2018) or which seasons (e.g. 1st, 2nd, 3rd, 4th) are you         │
│ referring to?                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
result = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content="yes to the overall best performance and other factors like win percentage no of trophies etc in ipl ")
        ]
    },
    config=thread
)



=== RAW TAVILY RESULTS ===
[{'query': 'official IPL team statistics all seasons total matches wins win percentages', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.iplt20.com/stats/ipl-standings', 'title': 'IPL ipl-standings Stats & Awards | All Time Records | IPLT20', 'content': '| 1 | ![Image 25](https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/playerimages/Vaibhav%20Sooryavanshi.png) [Vaibhav Sooryavanshi RR](https://www.iplt20.com/players/vaibhav-sooryavanshi/22203) ![Image 26](https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/teamlogos/RR.png) | 776 | 16 | 16 | 0 | 103 | 48.50 | 327 | 237.30 | 1 | 5 | 63 | 72 |. | 10 | ![Image 43](https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/playerimages/Dhruv%20Jurel.png) ![Image 44](https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/teamlogos/RR.png) [Dhruv Jurel RR](https://www.iplt20.com/players/dhruv-jurel/20620) | 515 | 16 | 16 

CancelledError: 

In [ ]:
format_messages(result["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ yes to the overall best performance and other factors like win percentage no of trophies etc in ipl             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ I understand you want a report on the overall best performance and other factors like win percentage, number of │
│ trophies in the Indian Premier League (IPL). I will start researching based on this information.                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Here is the final report: <think>                                                                               │
│ Okay, let me start by understanding the user's request. They want a comprehensive report on the best-performing │
│ team in the Indian Premier League (IPL) from 2008 to 2026, considering factors like win percentage, number of   │
│ trophies, and other metrics. The user prioritizes official sources and reputable sports sites.                  │
│                                                                                                                 │
│ First, I need to structure the report properly. The user provided findings that include win percentages,        │
│ trophies, additional metrics, head-to-head records, and team name changes. The answer should be well-organized  │
│ with sections like an introduction, key metrics, balanced analysis, and a conclusion. Each section needs to     │
│ reference sources correctly.                                                                                    │
│                                                                                                                 │
│ I should start by outlining the main sections. The introduction will set the context. Then, sections on win     │
│ percentages, trophies, head-to-head records, and other metrics. Each section should have subsections for        │
│ clarity. For example, under win percentages, list each team with their data.                                    │
│                                                                                                                 │
│ Next, I need to ensure all the data from the findings is included. For example, Gujarat Titans have the highest │
│ win percentage, and CSK and MI each have five trophies. I need to present this data factually. Also, mention    │
│ the head-to-head records between top teams like CSK vs KKR and MI vs CSK.                                       │
│                                                                                                                 │
│ The user emphasized using official IPL sources and reputable sites. The sources provided include ESPNcricinfo,  │
│ Cricbuzz, and IPL's official site. I need to cite these appropriately, using the [1], [2], etc., format as per  │
│ the citation rules.                                                                                             │
│                                                                                                                 │
│ I also need to check if there are any discrepancies in the data. For example, the user mentioned that KKR won 3 │
│ titles and MI and CSK five each. Also, RCB and GT have one title each. The head-to-head records are important   │
│ for context.                                                                                                    │
│                                                                                                                 │
│ I should verify if all the data from the findings is covered. For instance, the rebranding of Delhi Daredevils  │
│ to Delhi Capitals and Kochi Tuskers' folding. Including these adds depth to the report.                         │
│                                                                                                                 │
│ The conclusion needs to summarize the key points, reiterating which team is the best based on the metrics.      │
│ Here, MI and CSK are tied in trophies, but CSK has a better win percentage and head-to-head record against KKR. │
│                                                                                                                 │
│ I need to make sure the sources are listed correctly at the end, each with their corresponding numbers. The     │
│ user provided URLs and titles for each source, so I'll 